In [1]:
import os
import torch
import torchvision
from torch import nn
from d2l import torch as d2l
from torch.utils import data
import random
import numpy as np


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)
batch_size=68
epoch=5

In [2]:
# #@save
# d2l.DATA_HUB['hotdog'] = (d2l.DATA_URL + 'hotdog.zip',
#                          'fba480ffa8aa7e0febbb511d181409f899b9baa5')

# data_dir = d2l.download_extract('hotdog')

In [3]:
# 使用RGB通道的均值和标准差，以标准化每个通道
normalize = torchvision.transforms.Normalize(
    [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_augs = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    normalize])

test_augs = torchvision.transforms.Compose([
    torchvision.transforms.Resize([256, 256]),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    normalize])

In [4]:
data_dir='../data/hotdog'
train_imgs = torchvision.datasets.ImageFolder(os.path.join(data_dir, 'train'),transform=train_augs)
test_imgs = torchvision.datasets.ImageFolder(os.path.join(data_dir, 'test'),transform=test_augs)

In [5]:
import ssl
import certifi


def create_certifi_context():
    return ssl.create_default_context(
        cafile=certifi.where()
    )


ssl._create_default_https_context = (
    create_certifi_context
)

In [6]:
train_iter=data.DataLoader(train_imgs,batch_size=batch_size,shuffle=True)
test_iter=data.DataLoader(test_imgs,batch_size=batch_size,shuffle=False)

device=torch.device('cuda:0')
loss=nn.CrossEntropyLoss()
net=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT) #
net.fc=nn.Linear(net.fc.in_features,2)
nn.init.xavier_uniform_(net.fc.weight)
nn.init.zeros_(net.fc.bias)
net=net.to(device)

for parameter in net.parameters():
    parameter.requires_grad= False
for parameter in net.fc.parameters():
    parameter.requires_grad= True
trainer=torch.optim.SGD(net.fc.parameters(),lr=1e-3,weight_decay=0.001)





In [7]:
for i in range(epoch):
    net.train()

    for module in net.modules():
        if isinstance(module,nn.BatchNorm2d):
            module.eval() ##防止数据干扰已经计算好的running参数
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for x, y in train_iter:
        x = x.to(
                device,
                non_blocking=True
        )

        y = y.to(
                device,
                non_blocking=True
        )

        trainer.zero_grad()

        logits = net(x)
        l = loss(logits, y)

        l.backward()
        trainer.step()

        batch_count = len(y)

        train_loss += (
                l.item() * batch_count
        )

        train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        train_count += batch_count

    net.eval()

    test_loss = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for x, y in test_iter:
            x = x.to(
                    device,
                    non_blocking=True
            )

            y = y.to(
                    device,
                    non_blocking=True
            )

            logits = net(x)
            l = loss(logits, y)

            batch_count = len(y)

            test_loss += (
                l.item() * batch_count
            )

            test_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

            test_count += batch_count

        print(
            f"epoch {i + 1}，"
            f"train loss："
            f"{train_loss / train_count:.5f}，"
            f"train acc："
            f"{train_correct / train_count:.5f}"
        )

        print(
            f"epoch {i + 1}，"
            f"test loss："
            f"{test_loss / test_count:.5f}，"
            f"test acc："
            f"{test_correct / test_count:.5f}"
        )

epoch 1，train loss：0.45432，train acc：0.79200
epoch 1，test loss：0.43338，test acc：0.79750
epoch 2，train loss：0.41333，train acc：0.81650
epoch 2，test loss：0.38362，test acc：0.81750
epoch 3，train loss：0.37282，train acc：0.84300
epoch 3，test loss：0.35039，test acc：0.84875
epoch 4，train loss：0.34895，train acc：0.85650
epoch 4，test loss：0.32212，test acc：0.86500
epoch 5，train loss：0.31936，train acc：0.87000
epoch 5，test loss：0.30638，test acc：0.86750


In [8]:
for parameter in net.parameters():
    parameter.requires_grad=True
backbone_parameters=[]
for name,parameters in net.named_parameters():
    if not name.startswith('fc.'):
        backbone_parameters.append(parameters)
trainer=torch.optim.SGD([
    {
        "params":backbone_parameters,
        "lr":1e-4
    },
    {
        "params":net.fc.parameters(),
        "lr":1e-3
    }
    
],weight_decay=1e-3)


for i in range(epoch):
    net.train()

    
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for x, y in train_iter:
        x = x.to(
                device,
                non_blocking=True
        )

        y = y.to(
                device,
                non_blocking=True
        )

        trainer.zero_grad()

        logits = net(x)
        l = loss(logits, y)

        l.backward()
        trainer.step()

        batch_count = len(y)

        train_loss += (
                l.item() * batch_count
        )

        train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        train_count += batch_count

    net.eval()

    test_loss = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for x, y in test_iter:
            x = x.to(
                    device,
                    non_blocking=True
            )

            y = y.to(
                    device,
                    non_blocking=True
            )

            logits = net(x)
            l = loss(logits, y)

            batch_count = len(y)

            test_loss += (
                l.item() * batch_count
            )

            test_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

            test_count += batch_count

        print(
            f"epoch {i + 1}，"
            f"train loss："
            f"{train_loss / train_count:.5f}，"
            f"train acc："
            f"{train_correct / train_count:.5f}"
        )

        print(
            f"epoch {i + 1}，"
            f"test loss："
            f"{test_loss / test_count:.5f}，"
            f"test acc："
            f"{test_correct / test_count:.5f}"
        )
    

epoch 1，train loss：0.54559，train acc：0.72400
epoch 1，test loss：0.44573，test acc：0.79125
epoch 2，train loss：0.46300，train acc：0.77750
epoch 2，test loss：0.43476，test acc：0.80250
epoch 3，train loss：0.43193，train acc：0.79500
epoch 3，test loss：0.40043，test acc：0.80750
epoch 4，train loss：0.41002，train acc：0.81150
epoch 4，test loss：0.37846，test acc：0.81625
epoch 5，train loss：0.39170，train acc：0.82250
epoch 5，test loss：0.36028，test acc：0.82625


In [9]:
#接下来新建net 直接解冻训练10轮 与上面进行对比

In [10]:
net=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT) #
net.fc=nn.Linear(net.fc.in_features,2)
nn.init.xavier_uniform_(net.fc.weight)
nn.init.zeros_(net.fc.bias)
net=net.to(device)

for param in net.parameters():
    param.requires_grad=True
bone=[]
for name,param in net.named_parameters():
    if not name.startswith('fc.'):
        bone.append(param)
trainer=torch.optim.SGD([
    {
        "params":bone,
        "lr":1e-4
    },
    {
        "params":net.fc.parameters(),
        "lr":1e-3
    }
],weight_decay=1e-3)

for i in range(10):
    net.train()

    
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for x, y in train_iter:
        x = x.to(
                device,
                non_blocking=True
        )

        y = y.to(
                device,
                non_blocking=True
        )

        trainer.zero_grad()

        logits = net(x)
        l = loss(logits, y)

        l.backward()
        trainer.step()

        batch_count = len(y)

        train_loss += (
                l.item() * batch_count
        )

        train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        train_count += batch_count

    net.eval()

    test_loss = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for x, y in test_iter:
            x = x.to(
                    device,
                    non_blocking=True
            )

            y = y.to(
                    device,
                    non_blocking=True
            )

            logits = net(x)
            l = loss(logits, y)

            batch_count = len(y)

            test_loss += (
                l.item() * batch_count
            )

            test_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

            test_count += batch_count

        print(
            f"epoch {i + 1}，"
            f"train loss："
            f"{train_loss / train_count:.5f}，"
            f"train acc："
            f"{train_correct / train_count:.5f}"
        )

        print(
            f"epoch {i + 1}，"
            f"test loss："
            f"{test_loss / test_count:.5f}，"
            f"test acc："
            f"{test_correct / test_count:.5f}"
        )
    


epoch 1，train loss：0.92032，train acc：0.55700
epoch 1，test loss：0.79453，test acc：0.57125
epoch 2，train loss：0.68304，train acc：0.62550
epoch 2，test loss：0.67856，test acc：0.65000
epoch 3，train loss：0.59682，train acc：0.69300
epoch 3，test loss：0.58714，test acc：0.70250
epoch 4，train loss：0.53237，train acc：0.73500
epoch 4，test loss：0.52527，test acc：0.74250
epoch 5，train loss：0.51581，train acc：0.74300
epoch 5，test loss：0.47892，test acc：0.76625
epoch 6，train loss：0.45800，train acc：0.78400
epoch 6，test loss：0.44304，test acc：0.79250
epoch 7，train loss：0.43714，train acc：0.79800
epoch 7，test loss：0.41580，test acc：0.80625
epoch 8，train loss：0.41072，train acc：0.81850
epoch 8，test loss：0.39417，test acc：0.82000
epoch 9，train loss：0.40704，train acc：0.82050
epoch 9，test loss：0.37126，test acc：0.83625
epoch 10，train loss：0.37449，train acc：0.84100
epoch 10，test loss：0.36059，test acc：0.84250


In [11]:
# 方案3 冻结5轮解冻5轮 但是解冻的时候不更新running

In [12]:
net=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT) #
net.fc=nn.Linear(net.fc.in_features,2)
nn.init.xavier_uniform_(net.fc.weight)
nn.init.zeros_(net.fc.bias)
net=net.to(device)

for parameter in net.parameters():
    parameter.requires_grad= False
for parameter in net.fc.parameters():
    parameter.requires_grad= True
trainer=torch.optim.SGD(net.fc.parameters(),lr=1e-3,weight_decay=0.001)
for i in range(epoch):
    net.train()

    for module in net.modules():
        if isinstance(module,nn.BatchNorm2d):
            module.eval() ##防止数据干扰已经计算好的running参数
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for x, y in train_iter:
        x = x.to(
                device,
                non_blocking=True
        )

        y = y.to(
                device,
                non_blocking=True
        )

        trainer.zero_grad()

        logits = net(x)
        l = loss(logits, y)

        l.backward()
        trainer.step()

        batch_count = len(y)

        train_loss += (
                l.item() * batch_count
        )

        train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        train_count += batch_count

    net.eval()

    test_loss = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for x, y in test_iter:
            x = x.to(
                    device,
                    non_blocking=True
            )

            y = y.to(
                    device,
                    non_blocking=True
            )

            logits = net(x)
            l = loss(logits, y)

            batch_count = len(y)

            test_loss += (
                l.item() * batch_count
            )

            test_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

            test_count += batch_count

        print(
            f"epoch {i + 1}，"
            f"train loss："
            f"{train_loss / train_count:.5f}，"
            f"train acc："
            f"{train_correct / train_count:.5f}"
        )

        print(
            f"epoch {i + 1}，"
            f"test loss："
            f"{test_loss / test_count:.5f}，"
            f"test acc："
            f"{test_correct / test_count:.5f}"
        )

epoch 1，train loss：0.58314，train acc：0.71000
epoch 1，test loss：0.62748，test acc：0.67250
epoch 2，train loss：0.50140，train acc：0.75800
epoch 2，test loss：0.54334，test acc：0.72875
epoch 3，train loss：0.45208，train acc：0.79500
epoch 3，test loss：0.48499，test acc：0.76500
epoch 4，train loss：0.42099，train acc：0.81800
epoch 4，test loss：0.44127，test acc：0.78750
epoch 5，train loss：0.36909，train acc：0.84200
epoch 5，test loss：0.40634，test acc：0.80625


In [13]:
for parameter in net.parameters():
    parameter.requires_grad=True
backbone_parameters=[]
for name,parameters in net.named_parameters():
    if not name.startswith('fc.'):
        backbone_parameters.append(parameters)
trainer=torch.optim.SGD([
    {
        "params":backbone_parameters,
        "lr":1e-4
    },
    {
        "params":net.fc.parameters(),
        "lr":1e-3
    }
    
],weight_decay=1e-3)


for i in range(epoch):
    net.train()
    for module in net.modules():
        if isinstance(module,nn.BatchNorm2d):
            module.eval() ##防止数据干扰已经计算好的running参数
    
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for x, y in train_iter:
        x = x.to(
                device,
                non_blocking=True
        )

        y = y.to(
                device,
                non_blocking=True
        )

        trainer.zero_grad()

        logits = net(x)
        l = loss(logits, y)

        l.backward()
        trainer.step()

        batch_count = len(y)

        train_loss += (
                l.item() * batch_count
        )

        train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        train_count += batch_count

    net.eval()

    test_loss = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for x, y in test_iter:
            x = x.to(
                    device,
                    non_blocking=True
            )

            y = y.to(
                    device,
                    non_blocking=True
            )

            logits = net(x)
            l = loss(logits, y)

            batch_count = len(y)

            test_loss += (
                l.item() * batch_count
            )

            test_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

            test_count += batch_count

        print(
            f"epoch {i + 1}，"
            f"train loss："
            f"{train_loss / train_count:.5f}，"
            f"train acc："
            f"{train_correct / train_count:.5f}"
        )

        print(
            f"epoch {i + 1}，"
            f"test loss："
            f"{test_loss / test_count:.5f}，"
            f"test acc："
            f"{test_correct / test_count:.5f}"
        )
    

epoch 1，train loss：0.33989，train acc：0.85550
epoch 1，test loss：0.34283，test acc：0.84875
epoch 2，train loss：0.29884，train acc：0.87750
epoch 2，test loss：0.29773，test acc：0.88000
epoch 3，train loss：0.26769，train acc：0.89150
epoch 3，test loss：0.26847，test acc：0.89250
epoch 4，train loss：0.25056，train acc：0.90650
epoch 4，test loss：0.24618，test acc：0.90000
epoch 5，train loss：0.23212，train acc：0.90450
epoch 5，test loss：0.22918，test acc：0.90500


In [14]:
# 方案4 考虑直接解冻10轮 并且冻结running


In [15]:
net=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT) #
net.fc=nn.Linear(net.fc.in_features,2)
nn.init.xavier_uniform_(net.fc.weight)
nn.init.zeros_(net.fc.bias)
net=net.to(device)

for param in net.parameters():
    param.requires_grad=True
bone=[]
for name,param in net.named_parameters():
    if not name.startswith('fc.'):
        bone.append(param)
trainer=torch.optim.SGD([
    {
        "params":bone,
        "lr":1e-4
    },
    {
        "params":net.fc.parameters(),
        "lr":1e-3
    }
],weight_decay=1e-3)

for i in range(10):
    net.train()
    for module in net.modules():
        if isinstance(module,nn.BatchNorm2d):
            module.eval() ##防止数据干扰已经计算好的running参数
    
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for x, y in train_iter:
        x = x.to(
                device,
                non_blocking=True
        )

        y = y.to(
                device,
                non_blocking=True
        )

        trainer.zero_grad()

        logits = net(x)
        l = loss(logits, y)

        l.backward()
        trainer.step()

        batch_count = len(y)

        train_loss += (
                l.item() * batch_count
        )

        train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        train_count += batch_count

    net.eval()

    test_loss = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for x, y in test_iter:
            x = x.to(
                    device,
                    non_blocking=True
            )

            y = y.to(
                    device,
                    non_blocking=True
            )

            logits = net(x)
            l = loss(logits, y)

            batch_count = len(y)

            test_loss += (
                l.item() * batch_count
            )

            test_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

            test_count += batch_count

        print(
            f"epoch {i + 1}，"
            f"train loss："
            f"{train_loss / train_count:.5f}，"
            f"train acc："
            f"{train_correct / train_count:.5f}"
        )

        print(
            f"epoch {i + 1}，"
            f"test loss："
            f"{test_loss / test_count:.5f}，"
            f"test acc："
            f"{test_correct / test_count:.5f}"
        )
    


epoch 1，train loss：0.52182，train acc：0.74250
epoch 1，test loss：0.37074，test acc：0.83750
epoch 2，train loss：0.37261，train acc：0.84100
epoch 2，test loss：0.28848，test acc：0.89500
epoch 3，train loss：0.30370，train acc：0.87400
epoch 3，test loss：0.24745，test acc：0.91250
epoch 4，train loss：0.26960，train acc：0.89500
epoch 4，test loss：0.22215，test acc：0.92250
epoch 5，train loss：0.24366，train acc：0.90450
epoch 5，test loss：0.20459，test acc：0.92375
epoch 6，train loss：0.22373，train acc：0.91600
epoch 6，test loss：0.19205，test acc：0.93000
epoch 7，train loss：0.20624，train acc：0.92400
epoch 7，test loss：0.18313，test acc：0.93000
epoch 8，train loss：0.21091，train acc：0.91850
epoch 8，test loss：0.17612，test acc：0.93250
epoch 9，train loss：0.19537，train acc：0.92250
epoch 9，test loss：0.16985，test acc：0.93750
epoch 10，train loss：0.19600，train acc：0.92500
epoch 10，test loss：0.16537，test acc：0.94125
